#**Fine-Tuning the Pubmed BERT model for PFDxtract Tool**

This notebook will go through the steps of how to use the corrected annotations in the Theme_analysis_human_corrected folder of the repository.
This folder contains the files in which the PFDxtract tool users have corrected annotations and sent them to the developers for improvement.

Files Needed:


*   File of corrected annotations- AI_and_human_annotations with column Matched Sentences and column HUMAN LABEL
*   Folder of the latest iterated Pubmed version
*   Testing Iteration 2 file
*   Training Iteration 2 file



# **Step One: Prepare files**

Add the new Matched Sentences and human label data to the training iteration 2 file under the existing column headings of Text and Label.

Locate the testing iteration 2 file from the folder named Fine-tuning Files and download it to the same folder as this notebook.
Finally in order to get the latest iteration of the model, search hugging face to download it. To find the latest model, go onto the pubmed_analysis.py file to find the repo name and search for this to find the correct folder and user. Download the model file and store it in the same folder as the fine-tuning files.

# **Step Two: Import libraries and Import data**

Unless you change the file name for training iteration 2 and testing iteration 2, keep this code the same.

In [ ]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.metrics import accuracy_score, f1_score
import os
# ===============================
# Load Data
# ===============================
with open("Training Iteration 2.csv", "r", encoding="cp1252", errors="replace") as f:
    train_df = pd.read_csv(f)

with open("Testing Iteration 2.csv", "r", encoding="cp1252", errors="replace") as f:
    test_df = pd.read_csv(f)

# Normalise labels (CRITICAL FIX)
train_df["Label"] = train_df["Label"].str.strip().str.lower()
test_df["Label"] = test_df["Label"].str.strip().str.lower()

le = LabelEncoder()
train_df["labels"] = le.fit_transform(train_df["Label"])
test_df = test_df[test_df["Label"].isin(le.classes_)]
test_df["labels"] = le.transform(test_df["Label"])

# **Step Three: Update Iteration Numbers and load tokeniser**


Update the save_dir to a new name, for efficiency change the number to one above the current. Change the previous model path to the current/latest model path, this should also be iterated by one.

In [ ]:
# Save NEW label encoder for Iteration n
save_dir = "./pubmedbert_theme_classifier_iteration4"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(le, os.path.join(save_dir, "label_encoder.pkl"))

# Convert to HF datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


# Load Iteration previous Model
previous_model_path = "./pubmedbert_theme_classifier_iteration3"

tokenizer = AutoTokenizer.from_pretrained(previous_model_path)

# Load config & update number of labels for Iteration 3
config = AutoConfig.from_pretrained(
    previous_model_path,
    num_labels=len(le.classes_)   # NEW updated class count
)

# Load model & rebuild classification head for new labels
model = AutoModelForSequenceClassification.from_pretrained(
    previous_model_path,
    config=config,
    ignore_mismatched_sizes=True
)

def tokenize_function(examples):
    return tokenizer(
        examples["Text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )


# **Step Four: Loading Data**

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

test_dataset = test_dataset.remove_columns(
    [col for col in test_dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


# **Step Five: Training Arguments and Trainer**

Update the output diirectory to be the new model name, add one to the current file name. This code will also compute the metrics as the training is carried out.

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./pubmedbert_theme_classifier_iteration4",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
)


# Metrics
def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    labels = p.label_ids
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}


# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# **Step Six: Train and Save model**

In [ ]:
# Train
trainer.train()
print("PubMedBERT Iteration fine-tuned successfully!")

# Save Iteration Model + Tokenizer
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

Now the model has been fine-tuned on the new updated data and the saved model can be updated in the tool.

# **Step Seven: Upload Model to Hugging Face and update in pubmed_analysis.py**

Upload the saved directory of the latest model onto hugging face. When uploading the directory to hugging face, ensure it contains the label encoder file. Name the repository username/repo-name. Finally update MODEL_NAME below to your username/repo-name. Everything else in the pubmed analysis file should stay the same.

In [ ]:
MODEL_NAME = "kvara03/pubmedbert_theme_classifier_iteration4"

# Load tokenizer + model directly from HF
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
lemmatizer = WordNetLemmatizer()

# Load classification pipeline
clf = pipeline("text-classification", model=MODEL_NAME, tokenizer=MODEL_NAME)

# Download label encoder from HF and load it
label_path = hf_hub_download(
    repo_id=MODEL_NAME,
    filename="label_encoder.pkl"
)

Now the new model will be used for further annotations.